In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_mnist = DataLoader(
    datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist),
    batch_size=64, shuffle=True
)

class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(-1, 16*4*4)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model = LeNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1):
    for images, labels in train_mnist:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print("LeNet Loss:", loss.item())

transform_cifar = transforms.Compose([
    transforms.Resize(64),   # FAST instead of 224
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)

small_dataset = Subset(full_dataset, range(5000))

train_cifar = DataLoader(small_dataset, batch_size=64, shuffle=True)

alexnet = models.alexnet(weights=None)
alexnet.classifier[6] = nn.Linear(4096, 10)

optimizer = optim.Adam(alexnet.parameters(), lr=0.0001)

for epoch in range(1):
    for images, labels in train_cifar:
        optimizer.zero_grad()
        outputs = alexnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print("AlexNet Loss:", loss.item())

vgg16 = models.vgg16(weights=None)
vgg16.classifier[6] = nn.Linear(4096, 10)

optimizer = optim.Adam(vgg16.parameters(), lr=0.0001)

for epoch in range(1):
    for images, labels in train_cifar:
        optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print("VGG Loss:", loss.item())

LeNet Loss: 0.01902015693485737
AlexNet Loss: 2.177457332611084
VGG Loss: 1.997321605682373
